In [1]:
# use conda deepseek_env for this

from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableParallel, RunnableBranch, RunnableLambda, RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_deepseek import ChatDeepSeek

load_dotenv()

True

In [13]:
# Wedding planner 
# 1. create a Wedding persona base on country
# 2. pick menu base on pakistan or korean country
# 3. select no of function 
# 4. Calculate menu cost in dollars based on the country 

model = ChatDeepSeek(model='deepseek-chat')

wedding_persona = ChatPromptTemplate.from_messages([ # <-- 1st
    ('system','You are great wedding persona maker, and always respond with shortest output'),
    ('human', 'Make a wedding persona for the {country} wedding')
])

pakistani_menu_maker = ChatPromptTemplate.from_messages([ # <-- 2nd in a branch 
    ('system','You are great menu maker for pakistani types of wedding, and always respond with shortest output'),
    ('human', 'Make a menu for this persona\n:{persona}')
])

korean_menu_maker = ChatPromptTemplate.from_messages([ # <-- 2nd in a branch
    ('system','You are great menu maker for korean types of wedding, and always respond with shortest output'),
    ('human', 'Make a menu for this persona\n:{persona}')
])

default_menu_maker = ChatPromptTemplate.from_messages([
    ('system','You are great menu maker for any types of wedding, and always respond with shortest output'),
    ('human', 'Make a menu for this persona\n:{persona}')
])

function_planner = ChatPromptTemplate.from_messages([ # <-- 3rd in parallel with the menu budget maker
    ('system','You are great wedding function palnner, and always respond with shortest output'),
    ('human', 'Plan number of functions for the following wedding persona\n:{persona} \nand add this menu{menu}')
])

menu_budget_planner = ChatPromptTemplate.from_messages([ # <-- 3rd in parallel with the function_planner
    ('system', 'You are an expert menu budget maker, and always respond with shortest output'),
    ('human','Make a budget in dollars for the following {country} wedding menu: {menu} ')
])

summary_generator = ChatPromptTemplate.from_messages([ # <-- 3rd in parallel with the function_planner
    ('system', 'You are a fantastic wedding summarizer, and always respond with shortest output'),
    ('human',"""Make a summary for the wedding with persona:\n{persona}\nthen getting the following menu:\n{menu}
            \nthen lastly summarize the functions:\n{functions}\n,and the total budget for the menu:{menu_budget}""")
])

In [14]:
print('persona input formate : ',wedding_persona.input_variables)
print('menu maker formate : ', pakistani_menu_maker.input_variables)
print('function planner input formate : ', function_planner.input_variables)
print('menu budget planner input formate : ', menu_budget_planner.input_variables)
print('Summary generator input formate : ',summary_generator.input_variables)

persona input formate :  ['country']
menu maker formate :  ['persona']
function planner input formate :  ['menu', 'persona']
menu budget planner input formate :  ['country', 'menu']
Summary generator input formate :  ['functions', 'menu', 'menu_budget', 'persona']


In [15]:
persona_model = wedding_persona | model | StrOutputParser()

branchs = RunnableBranch(
    (
        lambda x: 'pakistan' in str(x.get('country','')).lower(), pakistani_menu_maker | model | StrOutputParser(),
    ),
    (
        lambda x: 'korea' in str(x.get('country','')).lower(), korean_menu_maker | model | StrOutputParser(),
    ),
    default_menu_maker | model | StrOutputParser()
)

function_chain = function_planner | model | StrOutputParser()

menu_budget_chain = menu_budget_planner | model | StrOutputParser()

summary_model = summary_generator | model | StrOutputParser()

1. You can just write the code directly into the chain rather than first storing it into a variable. But I Believe this approch is much Neat
2. You can just run the above code and pass the output to the next Runnable

In [16]:
chain = (
    RunnableParallel({
        "persona": persona_model,
        "country": RunnablePassthrough(),
    }) |
    RunnableParallel({
        'menu': branchs, # <-- either pakistani or korean.
        'persona': lambda x: x['persona'],
        'country': lambda x: x['country'],
    }) |
    RunnableParallel({
        'functions': function_chain,
        'menu': lambda x: x['menu'],
        'menu_budget': menu_budget_chain,
        'persona': lambda x: x['persona'],
        'country': lambda x: x['country'],
    }) |
    summary_model
)

In [17]:
try:
    print(chain.invoke({'country':'pakistan'}))
except Exception as e:
    print("Error: ",str(e))

### **Ayesha & Farhan’s Royal Mehndi Wedding**  
**Theme:** Regal & Vibrant | **Colors:** Gold, Emerald Green, Deep Pink  

**Events:**  
1. **Mehndi** – Outdoor, dhol, henna, *Dahi Bhallay/Murgh Biryani/Gulab Jamun* ($750).  
2. **Baraat** – Horse entrance, brass band, *Seekh Kebabs/Jalebi* ($300).  
3. **Walima** – Ballroom feast, *Hyderabadi Biryani/Shahi Tukray* ($600).  

**Budget:** **$1,650**  
**Vibe:** Luxurious, energetic, traditional. 💫
